In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set working directory (change as needed)
import os
os.chdir('/content/drive/MyDrive/scRNA_analysis/')

In [ ]:
# Install necessary packages
!pip install -q scanpy numpy pandas leidenalg harmonypy ktplotspy
print('Packages installed successfully')

In [ ]:
pip install seaborn

In [ ]:
pip install leidenalg

In [ ]:
pip install harmonypy

In [ ]:
pip install --upgrade cellphonedb

In [ ]:
import os

path = os.getcwd()

print(path)

In [ ]:
!mkdir Results_0-1m

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=600, facecolor='white')

In [ ]:
results_file = 'Results_0-1m/Meta_Mix_0-1m.h5ad'  # the file that will store the analysis results

In [ ]:
results_file_all = 'Results_0-1m/Meta_Mix_all_0-1m.h5ad'

In [ ]:
import pandas as pd

# Read the file
df = pd.read_csv("GSE234774_rnaseq_features.txt.gz", sep="\t", header=None)
print(df.head())  # Check the first few rows

# If no issues, rename the file as is
df.to_csv("features.tsv.gz", sep="\t", index=False, header=False, compression="gzip")

In [ ]:
import pandas as pd

# File path
input_path = 'features.tsv.gz'  # Original file (before modification)

# Load the file
features = pd.read_csv(input_path, header=None, sep="\t")

# Pad columns to convert to 3-column format
features[1] = features[0]  # Copy column 1 to column 2
features[2] = "Gene Expression"  # Add column 3 with a fixed value

# Save the modified data
features.to_csv(input_path, sep="\t", index=False, header=False, compression="gzip")

print(f"Converted file saved to: {input_path}")


In [ ]:
import pandas as pd

# Load the file
df = pd.read_csv("GSE234774_rnaseq_barcodes.txt.gz", sep="\t", header=None)
print(df.head())  # Check the first few rows

# If there are no issues, rename as is
df.to_csv("barcodes.tsv.gz", sep="\t", index=False, header=False, compression="gzip")

In [ ]:
# ===== Cell 3: Data Loading =====
import scanpy as sc
import numpy as np
import pandas as pd

# Load preprocessed h5ad file
print("Loading data...")
adata = sc.read_h5ad('/content/drive/MyDrive/scRNA_analysis/filtered_data_Ver2.h5ad')

print(f"Loading complete: {adata.n_obs} cells, {adata.n_vars} genes")
print("\nData summary:")
print(adata)

# Start analysis from here!

In [ ]:
adata

In [ ]:
# Extract conditions (e.g., age_old_1) from the index to create a 'batch' column
adata.obs['batch'] = adata.obs.index.str.split('-').str[0]

# Get unique values of the created 'batch' column as a list
unique_batches = adata.obs['batch'].unique()

# Display the list
print("Unique batches:")
print(unique_batches)

In [ ]:
# Extract specific batches
# Specify multiple batches (e.g., 'batch_1' and 'batch_2')
batches_to_extract = ['uninjured_uninjured_10', 'uninjured_uninjured_2', 'uninjured_uninjured_3','timecourse_4d_5', 'timecourse_4d_6',
 'timecourse_4d_8', 'timecourse_7d_4', 'timecourse_7d_9', 'timecourse_14d_10', 'timecourse_14d_8', 'timecourse_14d_9', 'timecourse_1d_2', 'timecourse_1d_3', 'timecourse_1d_8', 'timecourse_1m_10', 'timecourse_1m_5', 'timecourse_1m_9', 'timecourse_2m_2',
 'timecourse_2m_3', 'timecourse_2m_9']

# Extract cells where the 'batch' column matches any of the specified batches
adata = adata[adata.obs['batch'].isin(batches_to_extract), :]


In [ ]:
adata

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=100, )

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('mT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
adata

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
adata

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

Actually do the filtering by slicing the `AnnData` object.

In [ ]:
adata = adata[adata.obs.n_genes_by_counts < 10000, :]
adata = adata[adata.obs.pct_counts_mt < 10, :]

In [ ]:
adata

Total-count normalize (library-size correct) the data matrix $\mathbf{X}$ to 10,000 reads per cell, so that counts become comparable among cells.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)

Logarithmize the data:

In [ ]:
sc.pp.log1p(adata)

In [ ]:
adata

In [ ]:
adata.write(results_file_all)

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

In [ ]:
sc.pl.highly_variable_genes(adata)

Set the `.raw` attribute of the AnnData object to the normalized and logarithmized raw gene expression for later use in differential testing and visualizations of gene expression. This simply freezes the state of the AnnData object.

In [ ]:
adata

In [ ]:
adata.raw = adata

In [ ]:
adata = adata[:, adata.var.highly_variable]

In [ ]:
adata

Regress out effects of total counts per cell and the percentage of mitochondrial genes expressed. Scale the data to unit variance.

In [ ]:
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

In [ ]:
adata

In [ ]:
sc.pp.scale(adata, max_value=10)

In [ ]:
adata

In [ ]:
adata.write(results_file)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
adata.write(results_file)

In [ ]:
adata

## Computing the neighborhood graph

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

## Embedding the neighborhood graph

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(adata, resolution=1.5)

Plot the clusters, which agree quite well with the result of Seurat.

In [ ]:
sc.pl.umap(adata, color=['leiden'])

In [ ]:
adata.write(results_file)

# Create heatmap for cell annotation

In [ ]:
# Install required packages
!pip install celltypist

import numpy as np
import pandas as pd
import scanpy as sc
import celltypist

In [ ]:
adata = sc.read(results_file)

In [ ]:
adata.obs


In [ ]:
# Check available models
available_models = celltypist.models.download_models()
print(available_models)

In [ ]:
# Specify CellTypist model
model = 'Mouse_Whole_Brain.pkl'  # Change as needed

# Annotate each leiden cluster
unique_clusters = adata.obs['leiden'].unique()
cluster_annotations = {}
cell_type_counts = {}

for cluster in unique_clusters:
    print(f"Annotating cluster {cluster}...")

    # Subset data for each cluster
    cluster_data = adata[adata.obs['leiden'] == cluster]

    # Annotation using CellTypist
    predictions = celltypist.annotate(cluster_data, model=model, majority_voting=True)

    # Get the most common cell type name
    cell_type = predictions.predicted_labels.value_counts().idxmax()

    # Make cell type name unique
    if cell_type not in cell_type_counts:
        cell_type_counts[cell_type] = 1
    else:
        cell_type_counts[cell_type] += 1

    unique_cell_type = f"{cell_type}_{cell_type_counts[cell_type]}"

    # Save annotation for the cluster
    cluster_annotations[cluster] = unique_cell_type

# Add annotations corresponding to `leiden` clusters to adata.obs
adata.obs['leiden_annotation'] = adata.obs['leiden'].map(cluster_annotations)

# Check annotation results
print(adata.obs['leiden_annotation'].value_counts())

In [ ]:
adata.write(results_file)

print("File successfully saved!")

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
adata = sc.read(results_file)


In [ ]:
import pandas as pd

leiden_annotation_mapping = (
    adata.obs[['leiden', 'leiden_annotation']]
    .drop_duplicates()
    .sort_values('leiden')
)
print("Leiden/cell type mapping:")
print(leiden_annotation_mapping)


In [ ]:
cluster_annotation_mapping = (
    adata.obs.groupby('leiden')['leiden_annotation']
    .apply(lambda x: x.value_counts().idxmax())
    .reset_index()
    .rename(columns={'leiden': 'Leiden', 'leiden_annotation': 'annotation'})
)

print("Leiden/celltype mapping per cluster:")
print(cluster_annotation_mapping)


In [ ]:
import scanpy as sc

sc.pl.umap(
    adata,
    color='leiden_annotation',
    legend_loc='right margin',
    title='UMAP with Cluster Annotations',

)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


def plot_umap_with_labels(adata, annotation_col='leiden_annotation'):
    fig, ax = plt.subplots(figsize=(15, 15))


    umap_coords = adata.obsm['X_umap']
    annotations = adata.obs[annotation_col]


    unique_annotations = annotations.unique()
    colors = plt.cm.get_cmap('tab20', len(unique_annotations))
    annotation_colors = {ann: colors(i) for i, ann in enumerate(unique_annotations)}


    for ann in unique_annotations:
        cluster_indices = annotations == ann
        cluster_coords = umap_coords[cluster_indices]
        ax.scatter(cluster_coords[:, 0], cluster_coords[:, 1], label=ann, s=10, alpha=0.7, color=annotation_colors[ann])


    for ann in unique_annotations:
        cluster_indices = annotations == ann
        cluster_coords = umap_coords[cluster_indices]
        cluster_center = cluster_coords.mean(axis=0)


        label_offset = np.array([0.02, 0.02])
        label_position = cluster_center + label_offset


        ax.annotate(
            ann,
            xy=cluster_center,
            xytext=label_position,
            textcoords='offset points',
            arrowprops=dict(arrowstyle='-', color='gray', lw=1),
            fontsize=8,
            ha='left'
        )


    ax.set_xlabel('UMAP1', fontsize=8)
    ax.set_ylabel('UMAP2', fontsize=8)
    ax.set_title('UMAP with Simplified Annotations', fontsize=14)
    ax.grid(False)
    ax.legend([], frameon=False)
    plt.tight_layout()
    plt.show()

# Run
plot_umap_with_labels(adata, annotation_col='leiden_annotation')


In [ ]:
# Extract conditions (e.g., age_old_1) from the index to create a 'batch' column
adata.obs['batch'] = adata.obs.index.str.split('-').str[0]

# Get unique values of the created 'batch' column as a list
unique_batches = adata.obs['batch'].unique()

# Display the list
print("Unique batches:")
print(unique_batches)

In [ ]:
import collections

annotation_counts = collections.Counter()

def parse_cluster_name(anno):
    """
    Example annotation: "leiden_'327 Oligo NN'_7"
    -> Split by "'" and take index=1: "327 Oligo NN"
    -> Consider parts[1] as cell type: "Oligo"
    """
    if not isinstance(anno, str) or "'" not in anno:
        return anno  # Return as is if not a string or not in expected format
    main_part = anno.split("'\")[1]  # => "327 Oligo NN"
    parts = main_part.split()       # => ["327", "Oligo", "NN"] etc.
    if len(parts) < 2:
        return anno
    return parts[1]  # Example: "Oligo"


def make_cluster_label_dict(annotations):
    """
    annotations: pd.Series or list, etc.
    1) Extract the set of unique cluster strings
    2) Extract the "cell type" for each (Oligo, Astro, etc.)
    3) Create a dict with sequential numbers for each identical cell type
    """
    unique_annos = set(annotations)
    label_dict = {}

    # Use Counter to increment count for each cell type
    annotation_counts = collections.Counter()

    for anno in sorted(unique_annos):
        # Extract cell type from cluster string anno
        cell_type = parse_cluster_name(anno)
        # Increment count
        annotation_counts[cell_type] += 1
        # Example: "Oligo-1", "Oligo-2", ...
        label_dict[anno] = f"{cell_type}_{annotation_counts[cell_type]}"

    return label_dict

# Example: adata.obs['leiden_annotation'] is in the format "leiden_'327 Oligo NN'_7"
original_annos = adata.obs['leiden_annotation']

# 1) Create a dictionary to map cluster strings to "Oligo-1" etc.
cluster_label_dict = make_cluster_label_dict(original_annos)

# 2) Perform batch conversion using this dictionary (e.g., Series.map() or .apply())
adata.obs['leiden_annotation_simplified'] = original_annos.map(cluster_label_dict)

# Verify
print(adata.obs[['leiden_annotation', 'leiden_annotation_simplified']].head(30))


In [ ]:
# Get count and list of unique types
unique_simplified_counts = adata.obs['leiden_annotation_simplified'].value_counts()

# Display results
print(f"Number of unique leiden_annotation_simplified types: {len(unique_simplified_counts)}")
print(unique_simplified_counts)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import to_hex

# Determine colors for each cell type
color_map = {
    "Oligo": sns.color_palette("Blues", 10),  # Oligo is blue
    "Microglia": sns.color_palette("Greens", 6),  # Microglia is green
    "SPVI-SPVC": sns.color_palette("Purples", 6),  # SPVI-SPVC is purple
    "Astro-NT": sns.color_palette("Reds", 3) + sns.color_palette("Set1", 3),  # Astro-NT is red + colorful
    "BAM": sns.color_palette("Oranges", 2),
    "OPC": sns.color_palette("coolwarm", 3),
    "ABC": sns.color_palette("magma", 3),
    "Endo": sns.color_palette("viridis", 2),
    "VLMC": sns.color_palette("cividis", 2),
    "Ependymal": sns.color_palette("YlGnBu", 2),
    "Peri": sns.color_palette("PiYG", 2),
    "Lymphoid": sns.color_palette("husl", 3),
    "RO-RPA": sns.color_palette("dark:#5A9", 1),
}

# Define colors for each cluster
cluster_names = adata.obs["leiden_annotation_simplified"].unique()
cluster_colors = {}

for cluster in cluster_names:
    for cell_type, palette in color_map.items():
        if cluster.startswith(cell_type):  # Assign color based on cell type
            index = sum([c.startswith(cell_type) for c in cluster_names]) - 1
            cluster_colors[cluster] = to_hex(palette[min(index, len(palette) - 1)])
            break
    else:
        cluster_colors[cluster] = to_hex(np.random.rand(3,))  # Random color for others

# Apply colors to AnnData
adata.uns["leiden_annotation_simplified_colors"] = [cluster_colors[c] for c in cluster_names]

# UMAP plot
fig, ax = plt.subplots(figsize=(15, 10))
sc.pl.umap(
    adata,
    color="leiden_annotation_simplified",
    legend_loc="on data",
    title="UMAP with Cluster Annotations",
    legend_fontsize=10,
    size=10,
    ax=ax
)

plt.show()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import to_hex
import matplotlib.cm as cm

# Define colors for each cluster
cluster_names = adata.obs["leiden_annotation_simplified"].unique()
num_clusters = len(cluster_names)

# Combine several color palettes
palette1 = sns.color_palette("tab20", n_colors=min(20, num_clusters))  # up to 20 colors
palette2 = sns.color_palette("Set3", n_colors=min(12, num_clusters))  # up to 12 colors
palette3 = sns.color_palette("Paired", n_colors=min(12, num_clusters))  # up to 12 colors
palette4 = [cm.turbo(i / num_clusters) for i in range(num_clusters)]  # continuous colormap

# Merge all palettes
combined_palette = palette1 + palette2 + palette3 + palette4
final_palette = combined_palette[:num_clusters]  # trim to the required number

# Assign a color to each cluster
cluster_colors = {cluster: to_hex(final_palette[i]) for i, cluster in enumerate(cluster_names)}

# Apply colors to AnnData
adata.uns["leiden_annotation_simplified_colors"] = [cluster_colors[c] for c in cluster_names]

# UMAP plot (SVG export-ready)
fig, ax = plt.subplots(figsize=(15, 10))
sc.pl.umap(
    adata,
    color="leiden_annotation_simplified",
    legend_loc="on data",
    title="UMAP with Cluster Annotations",
    legend_fontsize=10,
    size=10,
    ax=ax,
    show=False  # save without displaying
)

# Save as SVG file
svg_filename = "umap_clusters.svg"
plt.savefig(svg_filename, format="svg", dpi=500, bbox_inches="tight")

# Print the SVG file path for confirmation
print(f"Saved SVG file: {svg_filename}")


In [ ]:
# === Output cluster name and color mapping in a copy-pasteable format ===
print("\n=== Cluster Name and Color Code Mapping ===")
for cluster in cluster_names:
    print(f'"{cluster}": "{cluster_colors[cluster]}",')

In [ ]:
cluster_colors_updated = {
    # ── Astro-NT (vivid colors) ──────────────────
    "Astro-NT_1": "#7B68EE",
    "Astro-NT_2": "#FF69B4",
    "Astro-NT_3": "#2FEF2F",
    "Astro-NT_4": "#DC143C",
    "Astro-NT_5": "#00CED1",
    "Astro-NT_6": "#FFD700",

    # ── ABC / VLMC (fixed) ──────────────────────────
    "ABC_1":  "#0099FF",
    "ABC_2":  "#0033CC",
    "VLMC_1": "#FF8C00",

    # ── Microglia (pastel) ───────────────────────
    "Microglia_1": "#B7E0CA",
    "Microglia_2": "#9FC982",
    "Microglia_3": "#A28FBC",
    "Microglia_4": "#C98582",
    "Microglia_5": "#C9B7E0",
    "Microglia_6": "#D8D8A5",
    "Microglia_7": "#C9B482",
    "Microglia_8": "#B2D8D8",

    # ── Lymphoid (pastel) ────────────────────────
    "Lymphoid_1": "#82C9C9",
    "Lymphoid_2": "#D8A5B0",
    "Lymphoid_3": "#C9D8A5",
    "Lymphoid_4": "#D8BFA5",

    # ── Oligo (pastel) ───────────────────────────
    "Oligo_1":  "#BFCDD8",
    "Oligo_2":  "#C5BF9F",
    "Oligo_3":  "#82C99F",
    "Oligo_4":  "#C393D1",
    "Oligo_5":  "#A5CDD8",
    "Oligo_6":  "#AAD8A5",
    "Oligo_7":  "#86A5C4",
    "Oligo_8":  "#D8A5CC",
    "Oligo_9":  "#D19D93",
    "Oligo_10": "#B9C982",
    "Oligo_11": "#A5ABD8",
    "Oligo_12": "#82C982",
    "Oligo_13": "#D8CCA5",

    # ── OPC (pastel) ─────────────────────────────
    "OPC_1": "#D8C3BF",
    "OPC_2": "#8494AD",
    "OPC_3": "#AD9D84",
    "OPC_4": "#C08AA6",

    # ── Others (pastel) ──────────────────────────
    "Endo_1":      "#ACB793",
    "Ependymal_1": "#CDDBBC",
    "Ependymal_2": "#BCE0D8",
    "Peri_1":      "#84AD90",
    "Peri_2":      "#ADBC84",
    "RO-RPA_1":    "#9FC5BD",

    # ── SPVI-SPVC (pastel) ───────────────────────
    "SPVI-SPVC_1": "#AD8492",
    "SPVI-SPVC_2": "#93B7A9",
    "SPVI-SPVC_3": "#C1A3A7",
    "SPVI-SPVC_4": "#7AB7B4",
    "SPVI-SPVC_5": "#B47AB7",
    "SPVI-SPVC_6": "#A7B47A",
    "SPVI-SPVC_7": "#7A9AB4",

    # ── SPVC (pastel) ────────────────────────────
    "SPVC_1": "#BBCBB2",
    "SPVC_2": "#99B27F",
    "SPVC_3": "#B4A3C1",
    "SPVC_4": "#E0B7D9",
    "SPVC_5": "#B77A7A",
    "SPVC_6": "#B2BCCB",

    # ── PGRN-PARN-MDRN (pastel) ──────────────────
    "PGRN-PARN-MDRN_1": "#9BA0C9",
    "PGRN-PARN-MDRN_2": "#CBB2BC",
}

In [ ]:
# Register the colormap in Scanpy with the correct order
cluster_names_sorted = list(adata.obs["leiden_annotation_simplified"].cat.categories)
adata.uns["leiden_annotation_simplified_colors"] = [cluster_colors_updated.get(name, "#cccccc") for name in cluster_names_sorted]

# Redraw and save UMAP
fig, ax = plt.subplots(figsize=(15, 10))
sc.pl.umap(
    adata,
    color="leiden_annotation_simplified",
    legend_loc="on data",
    title="UMAP with Custom Colors",
    legend_fontsize=10,
    size=10,
    ax=ax,
    show=False  # Save without displaying
)
plt.savefig("umap_clusters_custom.svg", format="svg", dpi=500, bbox_inches="tight")
print("Saved SVG with custom colors: umap_clusters_custom.svg")
# Save a version without legend
fig, ax = plt.subplots(figsize=(15, 10))
sc.pl.umap(
    adata,
    color="leiden_annotation_simplified",
    legend_loc=None,  # <─ Set this to None!
    title="UMAP without Legend",
    size=10,
    ax=ax,
    show=False
)
plt.savefig("umap_clusters_custom_nolegend.svg", format="svg", dpi=500, bbox_inches="tight")
print("Saved custom color SVG without legend: umap_clusters_custom_nolegend.svg")

In [ ]:
# Register the colormap in Scanpy with the correct order
cluster_names_sorted = list(adata.obs["leiden_annotation_simplified"].cat.categories)
adata.uns["leiden_annotation_simplified_colors"] = [cluster_colors_updated.get(name, "#cccccc") for name in cluster_names_sorted]

# Redraw and save UMAP
fig, ax = plt.subplots(figsize=(15, 10))
sc.pl.umap(
    adata,
    color="leiden_annotation_simplified",
    legend_loc="on data",
    title="UMAP with Custom Colors",
    legend_fontsize=10,
    size=10,
    ax=ax,
    show=False
)
plt.savefig("umap_clusters_custom.svg", format="svg", dpi=500, bbox_inches="tight")
print("Saved SVG with custom colors: umap_clusters_custom.svg")
# Save a version without legend
fig, ax = plt.subplots(figsize=(15, 10))
sc.pl.umap(
    adata,
    color="leiden_annotation_simplified",
    legend_loc=None,  # <─ Set this to None!
    title="UMAP without Legend",
    size=10,
    ax=ax,
    show=False
)
plt.savefig("umap_clusters_custom_nolegend.svg", format="svg", dpi=500, bbox_inches="tight")
print("Saved custom color SVG without legend: umap_clusters_custom_nolegend.svg")

In [ ]:

# Register the colormap in Scanpy with the correct order
cluster_names_sorted = list(adata.obs["leiden_annotation_simplified"].cat.categories)
adata.uns["leiden_annotation_simplified_colors"] = [cluster_colors_updated.get(name, "#cccccc") for name in cluster_names_sorted]

# Redraw and save UMAP
fig, ax = plt.subplots(figsize=(15, 10))
sc.pl.umap(
    adata,
    color="leiden_annotation_simplified",
    legend_loc="on data",
    title="UMAP with Custom Colors",
    legend_fontsize=10,
    size=10,
    ax=ax,
    show=False
)
plt.savefig("umap_clusters_custom.svg", format="svg", dpi=500, bbox_inches="tight")
print("Saved SVG with custom colors: umap_clusters_custom.svg")
# Save a version without legend
fig, ax = plt.subplots(figsize=(15, 10))
sc.pl.umap(
    adata,
    color="leiden_annotation_simplified",
    legend_loc=None,  # <─ Set this to None!
    title="UMAP without Legend",
    size=10,
    ax=ax,
    show=False
)
plt.savefig("umap_clusters_custom_nolegend.svg", format="svg", dpi=500, bbox_inches="tight")
print("Saved custom color SVG without legend: umap_clusters_custom_nolegend.svg")

In [ ]:
adata.write(results_file)

print("File successfully saved!")

In [ ]:
import matplotlib.pyplot as plt

# Specify figure size (15 inches wide, 10 inches tall)
fig, ax = plt.subplots(figsize=(25, 20))

# Draw UMAP plot
sc.pl.umap(
    adata,
    color='leiden_annotation_simplified',  # color by cluster annotation
    legend_loc='right margin',                  # display labels directly on the data
    title='UMAP with Cluster Annotations',
    legend_fontsize=10,                    # legend font size
    size=10,                               # size of each point
    ax=ax,
    save='umap_cluster_annotation_all.svg',
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Specify figure size (15 inches wide, 10 inches tall)
fig, ax = plt.subplots(figsize=(25, 20))

# Draw UMAP plot
sc.pl.umap(
    adata,
    color='leiden_annotation_simplified',  # color by cluster annotation
    legend_loc='on data',                  # display labels directly on the data
    title='UMAP with Cluster Annotations',
    legend_fontsize=10,                    # legend font size
    size=10,                               # size of each point
    ax=ax,
)

plt.show()

In [ ]:
marker_genes = ['Col3a1','Adgrg1']

In [ ]:
sc.pl.dotplot(adata, marker_genes, groupby=['leiden_annotation_simplified'],save='dotplot_COL3A1.svg')


In [ ]:
marker_genes = ['Gfap','Aqp4','S100b','Nes']

In [ ]:
sc.pl.dotplot(adata, marker_genes, groupby=['leiden_annotation_simplified'],save='dotplot_As.svg')

In [ ]:
# Extract only cells whose "leiden_annotation_simplified" value starts with "Astro-NT_"
astro_mask = adata.obs["leiden_annotation_simplified"].str.startswith("Astro-NT_")
astro_adata = adata[astro_mask].copy()


In [ ]:
# Example: if labels such as "Astro-NT_1" are already in use, they can be used as is
sc.tl.rank_genes_groups(astro_adata, groupby="leiden_annotation_simplified", method="wilcoxon")


In [ ]:
# Plot
sc.pl.rank_genes_groups(astro_adata, n_genes=25, sharey=False)

# Get as a DataFrame
marker_df = sc.get.rank_genes_groups_df(astro_adata, group=None)  # all clusters
marker_df.head(10)


In [ ]:
adata = sc.read(results_file)

In [ ]:
marker_genes = ['Acan', 'Aldh1l1', 'Aldoc', 'Aqp4', 'Atp1b2', 'Axin2', 'C3', 'Cd14', 'Cd109', 'Cdh2', 'Chst11', 'Crym', 'Cryab', 'Csgalnact1', 'Ctnnb1', 'Emp1', 'Fkbp5', 'G0s2', 'Gap43', 'Gfap', 'Ggta1', 'Gja1', 'Gpr84', 'Gss', 'H2-T23', 'Mki67', 'Lcn2', 'Mfge8', 'Mmp2', 'Mmp13', 'Mt1', 'Nes', 'Podn', 'Plaur', 'S100a4', 'S100a10', 'S100b', 'Serpina3n', 'Serping1', 'Slit2', 'Sox9', 'Sparcl1', 'Sphk1', 'Steap4', 'Timp1', 'Tm4sf1', 'Vim', 'Xylt1']

In [ ]:
marker_genes2 = ['Gfap','Aqp4','S100b','Aldh1l1','Cdh2','Nes','Vim','Lcn2','C3','Cspg4','Emp1','Cd109','Sparcl1','Slc39a12','Slc35f1','Ank3','Nav2','Peak1','Spp1','Lyz2','Apod','Hbb-bs',]

In [ ]:
astro_adata.obs['leiden_annotation_simplified'] = astro_adata.obs['leiden_annotation_simplified'].astype('category')
astro_adata.obs['leiden_annotation_simplified'].cat.reorder_categories(
    ["Astro-NT_1", "Astro-NT_2", "Astro-NT_3", "Astro-NT_4", "Astro-NT_5", "Astro-NT_6",],
    ordered=True,
)

In [ ]:
sc.pl.dotplot(
    astro_adata,
    marker_genes,
    groupby="leiden_annotation_simplified",
    save="dotplot_AsOnly.svg"
)

In [ ]:
sc.pl.dotplot(astro_adata, marker_genes, groupby=['leiden_annotation_simplified'],categories_order=['Astro-NT_1', 'Astro-NT_2', 'Astro-NT_3', 'Astro-NT_4', 'Astro-NT_5', 'Astro-NT_6'], save='dotplot_AsOnly.svg')

In [ ]:
sc.pl.dotplot(
    astro_adata,
    marker_genes2,
    groupby="leiden_annotation_simplified",
    categories_order=['Astro-NT_1', 'Astro-NT_2', 'Astro-NT_3', 'Astro-NT_4', 'Astro-NT_5', 'Astro-NT_6'],  # specify the desired order as a list
    save="dotplot_As_Ver2..svg"
)

In [ ]:
sc.pl.dotplot(
    astro_adata,
    marker_genes2,
    groupby="leiden_annotation_simplified",
    categories_order=['Astro-NT_1', 'Astro-NT_2', 'Astro-NT_3', 'Astro-NT_4', 'Astro-NT_5', 'Astro-NT_6'],  # specify the desired order as a list
    save="dotplot_As_Ver2..svg"
)

In [ ]:
import scanpy as sc
import numpy as np

# Set output directory
sc.settings.figdir = '/content/drive/MyDrive/scRNA_analysis/figures_Ver2'

# Get the root cell
root_cluster = "Astro-NT_3"
root_cell_name = astro_adata.obs[astro_adata.obs["leiden_annotation_simplified"] == root_cluster].index[0]

# Get the index number
root_index = np.where(astro_adata.obs_names == root_cell_name)[0][0]
astro_adata.uns['iroot'] = root_index

print(f"Root cell: {root_cell_name} (index: {root_index})")

# Compute DPT
sc.tl.dpt(astro_adata)

# Plot - the save parameter is a suffix without extension
sc.pl.umap(
    astro_adata,
    color=["dpt_pseudotime", "leiden_annotation_simplified"],
    save="_pseudotime_As_Ver2.svg"  # prepend an underscore
)

print(f"Output directory: {sc.settings.figdir}/umap_pseudotime_As_Ver2..svg")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Check pseudotime distribution per cluster
plt.figure(figsize=(10, 5))
sns.violinplot(
    x=astro_adata.obs["leiden_annotation_simplified"],
    y=astro_adata.obs["dpt_pseudotime"],
    order=sorted(astro_adata.obs["leiden_annotation_simplified"].unique())
)
plt.xticks(rotation=45, ha='right')
plt.title("Pseudotime by Astro-NT subclusters")
plt.tight_layout()
plt.show()


In [ ]:
# Build the graph using subcluster information (e.g., 'leiden_annotation_simplified')
sc.tl.paga(astro_adata, groups='leiden_annotation_simplified')


In [ ]:
sc.pl.paga(astro_adata, threshold=0.01, show=True)


In [ ]:
import matplotlib.pyplot as plt
import os

save_dir = '/content/drive/MyDrive/scRNA_analysis/figures_Ver2'
os.makedirs(save_dir, exist_ok=True)

# PAGA plot
fig, ax = plt.subplots(figsize=(10, 8))

sc.pl.paga(
    astro_adata,
    threshold=0.01,
    ax=ax,
    show=False  # important!
)

plt.tight_layout()

# Save
svg_path = f'{save_dir}/paga_astro_nt.svg'
png_path = f'{save_dir}/paga_astro_nt.png'

plt.savefig(svg_path, format="svg", bbox_inches="tight", dpi=300)
plt.savefig(png_path, format="png", bbox_inches="tight", dpi=300)

print(f"✓ SVG saved: {svg_path}")
print(f"✓ PNG saved: {png_path}")

plt.show()
plt.close()

In [ ]:
sc.tl.paga(adata, groups=  'leiden_annotation_simplified')  # or your cluster column name
sc.pl.paga(
    adata,
    threshold=0.5,
    node_size_scale=1.5,   # adjust node size (e.g., 1.0 for smaller nodes)
    fontsize=6        # label font size
)
   # first look at the overall connection structure


In [ ]:
# Set pseudotime root
root_cluster = "Astro-NT_3"   # change as needed
root_cells = adata.obs[adata.obs['leiden_annotation_simplified'] == root_cluster].index

# Compute pseudotime
sc.tl.dpt(adata, n_dcs=10, min_group_size=0.01)
adata.uns['iroot'] = adata.obs_names.get_indexer([root_cells[0]])[0]
sc.tl.dpt(adata)


In [ ]:
# UMAP + PAGA Graph
sc.pl.paga(adata, plot=False)  # this aligns positions based on PAGA
sc.tl.umap(adata, init_pos='paga')  # reflect PAGA structure in UMAP

# UMAP + pseudotime + PAGA
sc.pl.umap(adata, color=['dpt_pseudotime'], edges=True, size=50)


In [ ]:
adata = sc.read(results_file)

# Helper functions from cell 9_JkwhteFo23
import collections
def parse_cluster_name(anno):
    if not isinstance(anno, str) or "'" not in anno:
        return anno
    main_part = anno.split("'")[1]
    parts = main_part.split()
    if len(parts) < 2:
        return anno
    return parts[1]

def make_cluster_label_dict(annotations):
    unique_annos = set(annotations)
    label_dict = {}
    annotation_counts = collections.Counter()
    for anno in sorted(unique_annos):
        cell_type = parse_cluster_name(anno)
        annotation_counts[cell_type] += 1
        label_dict[anno] = f"{cell_type}_{annotation_counts[cell_type]}"
    return label_dict

# Always re-create 'leiden_annotation_simplified' from 'leiden_annotation'
# This assumes 'leiden_annotation' is always present in the loaded adata.
if 'leiden_annotation' in adata.obs.columns:
    original_annos = adata.obs['leiden_annotation']
    cluster_label_dict = make_cluster_label_dict(original_annos)
    adata.obs['leiden_annotation_simplified'] = original_annos.map(cluster_label_dict)
else:
    raise KeyError("'leiden_annotation' column is missing. Cannot create 'leiden_annotation_simplified'.")

# Now proceed with creating cell_labels from leiden_annotation_simplified
adata.obs['cell_labels'] = adata.obs['leiden_annotation_simplified']

import re
# Define rules for categorizing batch names
batch_mapping = {
    'uninjured': r'uninjured_.*',
    'timecourse_7d': r'timecourse_7d_.*',
    'drug_chabc': r'drug_chabc_.*',
    'mechanism_contusion': r'mechanism_contusion_.*',
    'timecourse_4d': r'timecourse_4d_.*',
    'timecourse_1d': r'timecourse_1d_.*',
    'timecourse_14d': r'timecourse_14d_.*',
    'timecourse_1m': r'timecourse_1m_.*',
    'timecourse_2m': r'timecourse_2m_.*',
    # add more rules if needed
}

# Function that classifies a batch name into a category
def get_batch_category(batch):
    for category, pattern in batch_mapping.items():
        if re.match(pattern, batch):
            return category
    return batch  # return as is if no category matches

# Create new labels by subdividing the existing cell_labels per batch
adata.obs['cell_labels'] = [
    f"{get_batch_category(batch)}_{cell_label}"
    for batch, cell_label in zip(adata.obs['batch'], adata.obs['cell_labels'])
]

In [ ]:
adata.obs

In [ ]:
adata.write(results_file)

In [ ]:
adata = sc.read(results_file)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import re
import collections # Add this import

# Check/create save directory
save_dir = '/content/drive/MyDrive/scRNA_analysis/figures_Ver2'
os.makedirs(save_dir, exist_ok=True)

# --- Start: Added code to create 'cell_labels' ---
# Helper functions from cell 9_JkwhteFo23
def parse_cluster_name(anno):
    if not isinstance(anno, str) or "'" not in anno:
        return anno
    main_part = anno.split("'\")[1] # Corrected line
    parts = main_part.split()
    if len(parts) < 2:
        return anno
    return parts[1]

def make_cluster_label_dict(annotations):
    unique_annos = set(annotations)
    label_dict = {}
    annotation_counts = collections.Counter()
    for anno in sorted(unique_annos):
        cell_type = parse_cluster_name(anno)
        annotation_counts[cell_type] += 1
        label_dict[anno] = f"{cell_type}_{annotation_counts[cell_type]}"
    return label_dict

# Ensure 'leiden_annotation_simplified' exists
if 'leiden_annotation' in adata.obs.columns and 'leiden_annotation_simplified' not in adata.obs.columns:
    original_annos = adata.obs['leiden_annotation']
    cluster_label_dict = make_cluster_label_dict(original_annos)
    adata.obs['leiden_annotation_simplified'] = original_annos.map(cluster_label_dict)
elif 'leiden_annotation_simplified' not in adata.obs.columns:
    raise KeyError("'leiden_annotation_simplified' column is missing. Cannot create 'cell_labels'.")

# Now proceed with creating cell_labels from leiden_annotation_simplified
adata.obs['cell_labels'] = adata.obs['leiden_annotation_simplified']

# Define rules to categorize batch names
batch_mapping = {
    'uninjured': r'uninjured_.*',
    'timecourse_7d': r'timecourse_7d_.*',
    'drug_chabc': r'drug_chabc_.*',
    'mechanism_contusion': r'mechanism_contusion_.*',
    'timecourse_4d': r'timecourse_4d_.*',
    'timecourse_1d': r'timecourse_1d_.*',
    'timecourse_14d': r'timecourse_14d_.*',
    'timecourse_1m': r'timecourse_1m_.*',
    'timecourse_2m': r'timecourse_2m_.*',
}

# Function to categorize batch names
def get_batch_category(batch):
    for category, pattern in batch_mapping.items():
        if re.match(pattern, batch):
            return category
    return batch  # Return as is if no match

# Create new 'cell_labels' by subdividing existing 'cell_labels' for each batch
adata.obs['cell_labels'] = [
    f"{get_batch_category(batch)}_{cell_label}"
    for batch, cell_label in zip(adata.obs['batch'], adata.obs['cell_labels'])
]
# --- End: Added code to create 'cell_labels' ---

# Extract time point and cell type information
adata.obs["timepoint"] = adata.obs["batch"].str.extract(r"(timecourse_\d+[dm]|uninjured)").fillna("unknown_timepoint")

# Extract major cell type from 'cell_labels'
adata.obs["cell_major_type"] = adata.obs["cell_labels"].apply(
    lambda x: x.split('_')[2] if x.startswith('timecourse') else x.split('_')[1]
)
adata.obs["subcluster"] = adata.obs["cell_labels"]

# Calculate subcluster proportions within each cell type per time point
cell_type_cluster_ratios = (
    adata.obs.groupby(["timepoint", "cell_major_type", "subcluster"])
    .size()
    .groupby(level=[0, 1], group_keys=False)
    .apply(lambda x: x / x.sum())
    .to_frame(name="ratio")
    .reset_index()
)

# Data verification
print("Data sample:")
print(cell_type_cluster_ratios.head())
print(f"\nTotal rows: {len(cell_type_cluster_ratios)}")

# Specify time point order
order = ["uninjured", "timecourse_1d", "timecourse_4d", "timecourse_7d",
         "timecourse_14d", "timecourse_1m", "timecourse_2m"]
cell_type_cluster_ratios["timepoint"] = pd.Categorical(
    cell_type_cluster_ratios["timepoint"],
    categories=order,
    ordered=True
)

# Transform data for Stacked Bar Plot
pivot_data = cell_type_cluster_ratios.pivot(
    index=['cell_major_type', 'timepoint'],
    columns='subcluster',
    values='ratio'
).fillna(0)

# Pivot data verification
print("\nPivot data shape:", pivot_data.shape)
print("Pivot data sample:")
print(pivot_data.head())

# Draw graph - important: explicitly use fig and ax
fig, ax = plt.subplots(figsize=(18, 10))

# Plot directly on ax
pivot_data.plot(
    kind='bar',
    stacked=True,
    ax=ax,  # Important!
    colormap='tab20',
    legend=True
)

# Graph settings
ax.set_title("Cell Subcluster Composition by Timepoint", fontsize=14)
ax.set_ylabel("Proportion", fontsize=12)
ax.set_xlabel("Timepoint / Cell Type", fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend(title="Subclusters", bbox_to_anchor=(1.05, 1), loc='upper left')

# Tight layout
fig.tight_layout()

# Save (before plt.show()!)
svg_path = f'{save_dir}/cell_subcluster_composition.svg'
png_path = f'{save_dir}/cell_subcluster_composition.png'

fig.savefig(svg_path, format="svg", bbox_inches="tight", dpi=300)
fig.savefig(png_path, format="png", bbox_inches="tight", dpi=300)

print(f"\n✓ SVG saved: {svg_path}")
print(f"✓ PNG saved: {png_path}")

# Save verification
if os.path.exists(svg_path):
    size_kb = os.path.getsize(svg_path) / 1024
    print(f"✓ SVG file size: {size_kb:.2f} KB")
    if size_kb < 10:
        print("⚠️ File is too small (almost empty)")
else:
    print("❌ File not found")

# Finally, display
plt.show()

# Close figure
plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import re
import collections

# Check/create save directory
save_dir = '/content/drive/MyDrive/scRNA_analysis/figures_Ver2'
os.makedirs(save_dir, exist_ok=True)

# --- Create 'cell_labels' ---
def parse_cluster_name(anno):
    if not isinstance(anno, str) or "'" not in anno:
        return anno
    main_part = anno.split("'\")[1]
    parts = main_part.split()
    if len(parts) < 2:
        return anno
    return parts[1]

def make_cluster_label_dict(annotations):
    unique_annos = set(annotations)
    label_dict = {}
    annotation_counts = collections.Counter()
    for anno in sorted(unique_annos):
        cell_type = parse_cluster_name(anno)
        annotation_counts[cell_type] += 1
        label_dict[anno] = f"{cell_type}_{annotation_counts[cell_type]}"
    return label_dict

if 'leiden_annotation' in adata.obs.columns and 'leiden_annotation_simplified' not in adata.obs.columns:
    original_annos = adata.obs['leiden_annotation']
    cluster_label_dict = make_cluster_label_dict(original_annos)
    adata.obs['leiden_annotation_simplified'] = original_annos.map(cluster_label_dict)
elif 'leiden_annotation_simplified' not in adata.obs.columns:
    raise KeyError("'leiden_annotation_simplified' column is missing.")

adata.obs['cell_labels'] = adata.obs['leiden_annotation_simplified']

batch_mapping = {
    'uninjured': r'uninjured_.*',
    'timecourse_7d': r'timecourse_7d_.*',
    'drug_chabc': r'drug_chabc_.*',
    'mechanism_contusion': r'mechanism_contusion_.*',
    'timecourse_4d': r'timecourse_4d_.*',
    'timecourse_1d': r'timecourse_1d_.*',
    'timecourse_14d': r'timecourse_14d_.*',
    'timecourse_1m': r'timecourse_1m_.*',
    'timecourse_2m': r'timecourse_2m_.*',
}

def get_batch_category(batch):
    for category, pattern in batch_mapping.items():
        if re.match(pattern, batch):
            return category
    return batch

adata.obs['cell_labels'] = [
    f"{get_batch_category(batch)}_{cell_label}"
    for batch, cell_label in zip(adata.obs['batch'], adata.obs['cell_labels'])
]

# Extract time point and cell type information
adata.obs["timepoint"] = adata.obs["batch"].str.extract(r"(timecourse_\d+[dm]|uninjured)").fillna("unknown_timepoint")

adata.obs["cell_major_type"] = adata.obs["cell_labels"].apply(
    lambda x: x.split('_')[2] if x.startswith('timecourse') else x.split('_')[1]
)
adata.obs["subcluster"] = adata.obs["cell_labels"]

# ========================================
# Filter for Astro-NT subclusters only
# ========================================
astro_nt_mask = adata.obs["cell_major_type"] == "Astro-NT"
adata_astro = adata[astro_nt_mask].copy()

print(f"Number of Astro-NT cells: {astro_nt_mask.sum()}")
print("List of Astro-NT subclusters:")
print(sorted(adata_astro.obs["subcluster"].unique()))

# Calculate subcluster proportions per time point (since cell_major_type is fixed, aggregate by timepoint × subcluster)
cell_type_cluster_ratios = (
    adata_astro.obs.groupby(["timepoint", "subcluster"])
    .size()
    .groupby(level=0, group_keys=False)
    .apply(lambda x: x / x.sum())
    .to_frame(name="ratio")
    .reset_index()
)

print("\nData sample:")
print(cell_type_cluster_ratios.head())

# Specify time point order
order = ["uninjured", "timecourse_1d", "timecourse_4d", "timecourse_7d",
         "timecourse_14d", "timecourse_1m", "timecourse_2m"]

cell_type_cluster_ratios["timepoint"] = pd.Categorical(
    cell_type_cluster_ratios["timepoint"],
    categories=order,
    ordered=True
)

# Transform data for Stacked Bar Plot
pivot_data = cell_type_cluster_ratios.pivot(
    index='timepoint',
    columns='subcluster',
    values='ratio'
).fillna(0)

# Sort by timecourse order
pivot_data = pivot_data.sort_index()

print("\nPivot data shape:", pivot_data.shape)
print(pivot_data)

# ========================================
# Extract Astro-NT colors from cluster_color_updated
# ========================================
# Get colors corresponding to pivot_data columns (subclusters)
# Assumes cluster_color_updated keys correspond to subcluster names
# Example: cluster_color_updated = {'timecourse_1d_Astro-NT_1': '#XXXXXX', ...}

subcluster_colors = []
missing_colors = []

for sc in pivot_data.columns:
    if sc in cluster_colors_updated:
        subcluster_colors.append(cluster_colors_updated[sc])
    else:
        # Fallback search for different key formats
        # Try to match by the end part of the subcluster name (e.g., "Astro-NT_1")
        matched = None
        for key, color in cluster_colors_updated.items():
            if sc in key or key in sc:
                matched = color
                break
        if matched:
            subcluster_colors.append(matched)
        else:
            missing_colors.append(sc)
            subcluster_colors.append('#CCCCCC')  # Fallback to gray

if missing_colors:
    print(f"\n⚠️ The following subclusters were not found in cluster_color_updated (replaced with gray):")
    for sc in missing_colors:
        print(f"  - {sc}")

# ========================================
# Draw graph
# ========================================
fig, ax = plt.subplots(figsize=(10, 6))

pivot_data.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    color=subcluster_colors,
    legend=True,
    width=0.7
)

# Format X-axis labels for readability (e.g., timecourse_Xd -> Xd, uninjured -> uninjured)
xticklabels = [
    tp.replace("timecourse_", "").replace("_", " ") if isinstance(tp, str) else str(tp)
    for tp in pivot_data.index
]
ax.set_xticklabels(xticklabels, rotation=45, ha='right', fontsize=11)

ax.set_title("Astro-NT Subcluster Composition by Timepoint", fontsize=14)
ax.set_ylabel("Proportion", fontsize=12)
ax.set_xlabel("Timepoint", fontsize=12)
ax.set_ylim(0, 1)

# Simplify legend to the end part of the subcluster name (e.g., Astro-NT_1)
handles, labels = ax.get_legend_handles_labels()
short_labels = []
for lb in labels:
    # timecourse_Xd_Astro-NT_1 -> Astro-NT_1
    parts = lb.split('_')
    # Extract the "Astro-NT_number" part
    try:
        astro_idx = next(i for i, p in enumerate(parts) if 'Astro' in p)
        short_labels.append('_'.join(parts[astro_idx:]))
    except StopIteration:
        short_labels.append(lb)

ax.legend(handles, short_labels, title="Astro-NT Subclusters",
          bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

fig.tight_layout()

# Save
svg_path = f'{save_dir}/AstroNT_subcluster_composition.svg'
png_path = f'{save_dir}/AstroNT_subcluster_composition.png'
fig.savefig(svg_path, format="svg", bbox_inches="tight", dpi=300)
fig.savefig(png_path, format="png", bbox_inches="tight", dpi=300)
print(f"\n✓ SVG saved: {svg_path}")
print(f"✓ PNG saved: {png_path}")

if os.path.exists(svg_path):
    size_kb = os.path.getsize(svg_path) / 1024
    print(f"✓ SVG file size: {size_kb:.2f} KB")
    if size_kb < 10:
        print("⚠️ File is too small (almost empty)")
else:
    print("❌ File not found")

plt.show()
plt.close(fig)

In [ ]:
import os

# Display the current working directory
current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")


In [ ]:
# CPDB

In [ ]:
!pip install cellphonedb
!pip install scvelo

In [ ]:
!pip install mousipy

In [ ]:
import pandas as pd
import matplotlib.pyplot as pl
import scanpy as sc
import numpy as np
import scvelo as scv
from tqdm.notebook import tqdm

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:97% !important; }</style>"))

import sys
sys.path.append('../src/mousipy/')

import scvelo as scv
from mousipy import translate

In [ ]:
import inspect
print(inspect.getsource(translate))

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set working directory (change as needed)
import os
os.chdir('/content/drive/MyDrive/scRNA_analysis')

In [ ]:
results_file = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_0-1m.h5ad'
results_file_all = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_all_0-1m.h5ad'

In [ ]:
import os

# Directory path
directory_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/'

# Get the list of files in the directory
files_in_directory = os.listdir(directory_path)

# Display the file list
print(f"Files in directory '{directory_path}':\n{files_in_directory}")

In [ ]:
adata_all= sc.read(results_file_all)

In [ ]:
adata = sc.read(results_file)


In [ ]:
adata.obs

In [ ]:
print(adata.obs.columns.tolist())

In [ ]:
print(adata_all.obs.columns.tolist())

In [ ]:
annotations = adata.obs['leiden_annotation_simplified'].unique()
sorted_annotations = sorted(annotations)

print("\nSorted list of leiden_annotation_simplified:")
for annotation in sorted_annotations:
    print(f"- {annotation}")


In [ ]:
# Verify that both share the same cell index (barcodes)
# sort if needed
# e.g., you can reorder with adata_all = adata_all[adata_ver2.obs_names, :]
# however, if they were derived from the same base, the order is typically assumed to be the same
# Add Leiden annotation
adata_all.obs['leiden_annotation_simplified'] = adata.obs['leiden_annotation_simplified']

# Save the data with the new obs column added
adata_all.write(results_file_all)

In [ ]:
annotations = adata_all.obs['leiden_annotation_simplified'].unique()
sorted_annotations = sorted(annotations)

print("\nSorted list of leiden_annotation_simplified:")
for annotation in sorted_annotations:
    print(f"- {annotation}")


In [ ]:
adata = sc.read(results_file_all)

In [ ]:
adata

In [ ]:
print(adata.var)

In [ ]:
drop_columns = [1, 3, 4, 5, 6, 7, 8,]

In [ ]:
drop_columns = [i - 1 for i in drop_columns]

In [ ]:
drop_column_names = adata.var.columns[drop_columns]

In [ ]:
adata.var = adata.var.drop(drop_column_names, axis=1)

In [ ]:
print(adata.var)

In [ ]:
import numpy as np
import scanpy as sc
import gc

# (1) Number of splits (adjust depending on memory; try 4 first)
n_splits = 4
cell_indices = np.array_split(np.arange(adata.n_obs), n_splits)

translated_chunks = []

for i, idx in enumerate(cell_indices):
    print(f"Processing chunk {i+1}/{n_splits} ({len(idx)} cells)...")

    # extract chunk
    chunk = adata[idx].copy()

    # run translate
    translated_chunk = translate(chunk)
    del chunk
    gc.collect()

    # temporarily save to disk (to conserve memory)
    path = f'/content/drive/MyDrive/scRNA_analysis/chunk_{i}.h5ad'
    translated_chunk.write_h5ad(path)
    del translated_chunk
    gc.collect()
    print(f"  -> Saved: {path}")

# (2) Release original data
del adata
gc.collect()

# (3) Load chunks and concatenate
chunks = []
for i in range(n_splits):
    path = f'/content/drive/MyDrive/scRNA_analysis/chunk_{i}.h5ad'
    chunks.append(sc.read_h5ad(path))

adata = sc.concat(chunks, join='outer')
del chunks
gc.collect()

print("Done!")
print(adata)

In [ ]:
print(adata.var)

In [ ]:
adata.var['gene_ids'] = adata.var.index

In [ ]:
print(adata.var)

In [ ]:
drop_columns = [2]

In [ ]:
drop_columns = [i - 1 for i in drop_columns]

In [ ]:
drop_column_names = adata.var.columns[drop_columns]

In [ ]:
adata.var = adata.var.drop(drop_column_names, axis=1)

In [ ]:
print(adata.var)

In [ ]:
# Change 'leiden_annotation_simplified' to 'cell_labels'
adata.obs['cell_labels'] = adata.obs['leiden_annotation_simplified']
del adata.obs['leiden_annotation_simplified']  # Optionally delete the original column

In [ ]:
annotations = adata.obs['cell_labels'].unique()
sorted_annotations = sorted(annotations)

print("\nSorted list of cell_labels:")
for annotation in sorted_annotations:
    print(f"- {annotation}")


In [ ]:
import re

# Define rules to categorize batch names
batch_mapping = {
    'uninjured': r'uninjured_.*',
    'timecourse_7d': r'timecourse_7d_.*',
    'timecourse_4d': r'timecourse_4d_.*',
    'timecourse_1d': r'timecourse_1d_.*',
    'timecourse_14d': r'timecourse_14d_.*',
    'timecourse_1m': r'timecourse_1m_.*',
    # Add more rules if needed
}

# Function to categorize batch names
def get_batch_category(batch):
    for category, pattern in batch_mapping.items():
        if re.match(pattern, batch):
            return category
    return batch  # Return as is if no match

# Create new 'cell_labels' by subdividing existing 'cell_labels' for each batch
adata.obs['cell_labels'] = [
    f"{get_batch_category(batch)}_{cell_label}"
    for batch, cell_label in zip(adata.obs['batch'], adata.obs['cell_labels'])
]

# Verify new cell_labels
print(adata.obs.head())

In [ ]:
annotations = adata.obs['cell_labels'].unique()
sorted_annotations = sorted(annotations)

print("\nSorted list of cell_labels:")
for annotation in sorted_annotations:
    print(f"- {annotation}")


In [ ]:
adata.write('Results_0-1m/Meta_Mix_CPDB.h5ad')

In [ ]:
annotations = adata_all.obs['leiden_annotation_simplified'].unique()
sorted_annotations = sorted(annotations)

print("\nSorted list of leiden_annotation_simplified:")
for annotation in sorted_annotations:
    print(f"- {annotation}")


In [ ]:
import pandas as pd
import glob
import os

In [ ]:
cpdb_input_dir = 'db'
os.listdir(cpdb_input_dir)

In [ ]:
print(sys.version)

In [ ]:
import pandas as pd
import sys
import os

pd.set_option('display.max_columns', 100)

# Define our base directory for the analysis
original_cwd = os.getcwd() # Save original working directory
os.chdir('db')

In [ ]:
# -- Version of the databse
cpdb_version = 'v5.0.0'

# -- Path where the input files to generate the database are located
cpdb_target_dir = os.path.join(cpdb_input_dir, cpdb_version)

In [ ]:
# from cellphonedb.utils import db_utils

# db_utils.download_database(cpdb_target_dir, cpdb_version)

os.chdir(original_cwd) # Restore original working directory

In [ ]:
# from cellphonedb.utils import db_utils

# db_utils.create_db(cpdb_input_dir)

os.chdir(original_cwd) # Restore original working directory

In [ ]:
# Extract specific batches
# Specify multiple batches (e.g., 'batch_1' and 'batch_2')
batches_to_extract = ['timecourse_4d_5', 'timecourse_4d_6', 'timecourse_4d_8',]

# Extract cells where the 'batch' column matches any of the specified batches
adata = adata[adata.obs['batch'].isin(batches_to_extract), :]

In [ ]:
adata

In [ ]:
adata.write('/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_4d_selected.h5ad')

# Check if the file was actually saved
import os
file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_4d_selected.h5ad'
if os.path.exists(file_path):
    print(f"File successfully written to: {file_path}")
else:
    print(f"Error: File was NOT written to: {file_path}")

In [ ]:
annotations = adata.obs['cell_labels'].unique()
sorted_annotations = sorted(annotations)

print("\nSorted list of cell_labels:")
for annotation in sorted_annotations:
    print(f"- {annotation}")


In [ ]:
df_meta = pd.DataFrame(data={'Cell':list(adata.obs.index),
                             'cell_type':[ i for i in adata.obs['cell_labels']]
                            })
df_meta.set_index('Cell', inplace=True)
df_meta.to_csv('metadata_4d.tsv', sep = '\t')

In [ ]:
print("Unique values of df_meta['cell_type']:")
print(df_meta['cell_type'].unique())

In [ ]:
cpdb_file_path = '/content/drive/MyDrive/scRNA_analysis/db/cellphonedb.zip'
meta_file_path = '/content/drive/MyDrive/scRNA_analysis/db/metadata_4d.tsv'
counts_file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_4d_selected.h5ad'
out_path = '/content/drive/MyDrive/scRNA_analysis/figures_Ver2'

### Plotting CellPhoneDB results


In [ ]:
metadata = pd.read_csv(meta_file_path, sep = '\t')
metadata.head(100)

In [ ]:
import anndata

adata = anndata.read_h5ad(counts_file_path)
adata.shape

In [ ]:
list(adata.obs.index).sort() == list(metadata['Cell']).sort()

In [ ]:
pip show cellphonedb

In [ ]:
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

cpdb_results = cpdb_statistical_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,             # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                       # optional: whether to score interactions or not.
    iterations = 1000,                               # denotes the number of shufflings performed in the analysis.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 5,                                     # number of threads to use in the analysis.
    debug_seed = 42,                                 # debug randome seed. To disable >=0.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    pvalue = 0.05,                                   # P-value threshold to employ for significance.
    subsampling = False,                             # To enable subsampling the data (geometri sketching).
    subsampling_log = True,                         # (mandatory) enable subsampling log1p for non log-transformed data inputs.
    subsampling_num_pc = 100,                        # Number of componets to subsample via geometric skectching (dafault: 100).
    subsampling_num_cells = 1000,                    # Number of cells to subsample (integer) (default: 1/3 of the dataset).
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = out_path,                          # Path to save results.
    output_suffix = "timecourse_4d"                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

In [ ]:
print(cpdb_results.keys())

In [ ]:
print(adata.obs.columns)


In [ ]:
!pip install anndata

In [ ]:
!pip install ktplotspy

In [ ]:
import anndata as ad
import pandas as pd
import ktplotspy as kpy
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
from pathlib import Path
import pandas as pd
import anndata as ad
import glob

# Define data directories
DATA_DIR = Path("/content/drive/MyDrive/scRNA_analysis/figures_Ver2")
RESULTS_DIR = Path("/content/drive/MyDrive/scRNA_analysis/figures_Ver2")

# 1) Load the .h5ad file
adata = ad.read_h5ad("/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_4d_selected.h5ad")

# 2) Get the most recent CellPhoneDB output files
def get_latest_file(directory, pattern):
    """Return the most recent file in the specified directory."""
    files = list(directory.glob(pattern))
    if files:
        return max(files, key=lambda f: f.stat().st_mtime)  # sort by last-modified time
    else:
        return None

# Pick the most recent file for each output type
means_file = get_latest_file(RESULTS_DIR, "statistical_analysis_interaction_scores_*.txt")
pvals_file = get_latest_file(RESULTS_DIR, "statistical_analysis_pvalues_*.txt")
decon_file = get_latest_file(RESULTS_DIR, "statistical_analysis_deconvoluted_*.txt")
score_file = get_latest_file(RESULTS_DIR, "statistical_analysis_interaction_scores_*.txt")



# Usage

# 3) Load the files
if means_file:
    means = pd.read_csv(means_file, sep="\t")
    print(f"Loaded means file: {means_file.name}")
else:
    print("No means file found.")

if pvals_file:
    pvals = pd.read_csv(pvals_file, sep="\t")
    print(f"Loaded p-values file: {pvals_file.name}")
else:
    print("No p-values file found.")

if decon_file:
    decon = pd.read_csv(decon_file, sep="\t")
    print(f"Loaded deconvoluted file: {decon_file.name}")
else:
    print("No deconvoluted file found.")

if score_file:
    interaction_scores = pd.read_csv(score_file, sep="\t")
    print(f"Loaded score file: {decon_file.name}")
else:
    print("No score file found.")

# Check data shapes
print("adata shape:", adata.shape)
print("means shape:", means.shape if 'means' in locals() else "No means data")
print("pvals shape:", pvals.shape if 'pvals' in locals() else "No p-values data")
print("decon shape:", decon.shape if 'decon' in locals() else "No deconvoluted data")



In [ ]:
kpy.plot_cpdb_heatmap(pvals=pvals, figsize=(15, 15), title="Sum of significant interactions")

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# Helper function to identify metadata columns
def identify_metadata_cols(df):
    """Identify metadata columns by looking for '|' in column names."""
    metadata_cols = []
    for col in df.columns:
        if '|' in str(col):
            break
        metadata_cols.append(col)
    return metadata_cols

# Clean means DataFrame
means_metadata_cols = identify_metadata_cols(means)
means_metadata = means[means_metadata_cols].copy()
means_numeric = means.drop(columns=means_metadata_cols).apply(pd.to_numeric, errors='coerce').fillna(0)
means_clean = pd.concat([means_metadata, means_numeric], axis=1)

# Clean pvals DataFrame
pvals_metadata_cols = identify_metadata_cols(pvals)
pvals_metadata = pvals[pvals_metadata_cols].copy()
pvals_numeric = pvals.drop(columns=pvals_metadata_cols).apply(pd.to_numeric, errors='coerce').fillna(1)
pvals_clean = pd.concat([pvals_metadata, pvals_numeric], axis=1)

# Clean interaction_scores DataFrame if it exists
interaction_scores_clean = None
if 'interaction_scores' in locals() and interaction_scores is not None:
    scores_metadata_cols = identify_metadata_cols(interaction_scores)
    scores_metadata = interaction_scores[scores_metadata_cols].copy()
    scores_numeric = interaction_scores.drop(columns=scores_metadata_cols).apply(pd.to_numeric, errors='coerce').fillna(0)
    interaction_scores_clean = pd.concat([scores_metadata, scores_numeric], axis=1)

# Define a directory for saving figures
save_dir = '/content/drive/MyDrive/scRNA_analysis/figures_Ver2'
os.makedirs(save_dir, exist_ok=True)

# Call kpy.plot_cpdb and capture the returned figure object
fig_cpdb = kpy.plot_cpdb(
    adata=adata,
    cell_type1="timecourse_4d_Astro-NT_1|timecourse_4d_Astro-NT_2|timecourse_4d_Astro-NT_3|timecourse_4d_Astro-NT_4|timecourse_4d_Astro-NT_5",
    cell_type2="timecourse_4d_VLMC_1|timecourse_4d_VLMC_2|timecourse_4d_ABC_1",
    means=means_clean, # Use the cleaned DataFrame
    pvals=pvals_clean, # Use the cleaned DataFrame
    celltype_key="cell_labels",
    genes=["COL3A1"],
    figsize=(18, 6),
    title="Interactions ",
    max_size=6,
    highlight_size=0.75,
    degs_analysis=False,
    standard_scale=True,
    interaction_scores=interaction_scores_clean, # Use the cleaned DataFrame
    scale_alpha_by_interaction_scores=True,
    min_interaction_score=20,
)

# Construct file paths
svg_path = os.path.join(save_dir, "cpdb_interactions_4d_COL3A1.svg")
png_path = os.path.join(save_dir, "cpdb_interactions_4d_COL3A1.png")

# Save the figure using ggplot's save method
if fig_cpdb:
    fig_cpdb.save(svg_path, format="svg", dpi=300)
    fig_cpdb.save(png_path, format="png", dpi=300)
    print(f"Plot saved as SVG: {svg_path}")
    print(f"Plot saved as PNG: {png_path}")
    plt.close() # Close any underlying matplotlib figure if created by plotnine
else:
    print("kpy.plot_cpdb did not return a Figure object, or returned None. Cannot save directly.")

In [ ]:
# timecourse_0d Version

In [ ]:
import scanpy as sc

# Load the file
adata = sc.read_h5ad('/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB.h5ad')

In [ ]:
# Extract specific batches
# Specify multiple batches (e.g., 'batch_1' and 'batch_2')
batches_to_extract = ['uninjured_uninjured_10', 'uninjured_uninjured_2', 'uninjured_uninjured_3',]

# Extract cells where the 'batch' column matches any of the specified batches
adata = adata[adata.obs['batch'].isin(batches_to_extract), :]

In [ ]:
adata.write('/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_0d_selected.h5ad')

# Check if the file was actually saved
import os
file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_0d_selected.h5ad'
if os.path.exists(file_path):
    print(f"File successfully written to: {file_path}")
else:
    print(f"Error: File was NOT written to: {file_path}")

In [ ]:
df_meta = pd.DataFrame(data={'Cell':list(adata.obs.index),
                             'cell_type':[ i for i in adata.obs['cell_labels']]
                            })
df_meta.set_index('Cell', inplace=True)
df_meta.to_csv('metadata_0d.tsv', sep = '\t')

In [ ]:
print("Unique values of df_meta['cell_type']:")
print(df_meta['cell_type'].unique())

In [ ]:
cpdb_file_path = '/content/drive/MyDrive/scRNA_analysis/db/cellphonedb.zip'
meta_file_path = '/content/drive/MyDrive/scRNA_analysis/db/metadata_0d.tsv'
counts_file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_0d_selected.h5ad'
out_path = '/content/drive/MyDrive/scRNA_analysis/figures_Ver2'

In [ ]:
import anndata

adata = anndata.read_h5ad(counts_file_path)
adata.shape

In [ ]:
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

cpdb_results = cpdb_statistical_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,             # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                       # optional: whether to score interactions or not.
    iterations = 1000,                               # denotes the number of shufflings performed in the analysis.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 5,                                     # number of threads to use in the analysis.
    debug_seed = 42,                                 # debug randome seed. To disable >=0.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    pvalue = 0.05,                                   # P-value threshold to employ for significance.
    subsampling = False,                             # To enable subsampling the data (geometri sketching).
    subsampling_log = True,                         # (mandatory) enable subsampling log1p for non log-transformed data inputs.
    subsampling_num_pc = 100,                        # Number of componets to subsample via geometric skectching (dafault: 100).
    subsampling_num_cells = 1000,                    # Number of cells to subsample (integer) (default: 1/3 of the dataset).
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = out_path,                          # Path to save results.
    output_suffix = "timecourse_0d"                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

In [ ]:
# timecourse_1d Version

In [ ]:
import scanpy as sc

# Load the file
adata = sc.read_h5ad('/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB.h5ad')

In [ ]:
# Extract specific batches
# Specify multiple batches (e.g., 'batch_1' and 'batch_2')
batches_to_extract = ['timecourse_1d_2', 'timecourse_1d_3', 'timecourse_1d_8',]

# Extract cells where the 'batch' column matches any of the specified batches
adata = adata[adata.obs['batch'].isin(batches_to_extract), :]

In [ ]:
adata.write('/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_1d_selected.h5ad')

# Check if the file was actually saved
import os
file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_1d_selected.h5ad'
if os.path.exists(file_path):
    print(f"File successfully written to: {file_path}")
else:
    print(f"Error: File was NOT written to: {file_path}")

In [ ]:
df_meta = pd.DataFrame(data={'Cell':list(adata.obs.index),
                             'cell_type':[ i for i in adata.obs['cell_labels']]
                            })
df_meta.set_index('Cell', inplace=True)
df_meta.to_csv('metadata_1d.tsv', sep = '\t')

In [ ]:
print("Unique values of df_meta['cell_type']:")
print(df_meta['cell_type'].unique())

In [ ]:
cpdb_file_path = '/content/drive/MyDrive/scRNA_analysis/db/cellphonedb.zip'
meta_file_path = '/content/drive/MyDrive/scRNA_analysis/db/metadata_1d.tsv'
counts_file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_1d_selected.h5ad'
out_path = '/content/drive/MyDrive/scRNA_analysis/figures_Ver2'

In [ ]:
import anndata

adata = anndata.read_h5ad(counts_file_path)
adata.shape

In [ ]:
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

cpdb_results = cpdb_statistical_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,             # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                       # optional: whether to score interactions or not.
    iterations = 1000,                               # denotes the number of shufflings performed in the analysis.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 5,                                     # number of threads to use in the analysis.
    debug_seed = 42,                                 # debug randome seed. To disable >=0.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    pvalue = 0.05,                                   # P-value threshold to employ for significance.
    subsampling = False,                             # To enable subsampling the data (geometri sketching).
    subsampling_log = True,                         # (mandatory) enable subsampling log1p for non log-transformed data inputs.
    subsampling_num_pc = 100,                        # Number of componets to subsample via geometric skectching (dafault: 100).
    subsampling_num_cells = 1000,                    # Number of cells to subsample (integer) (default: 1/3 of the dataset).
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = out_path,                          # Path to save results.
    output_suffix = "timecourse_1d"                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

In [ ]:
# timecourse_7d Version

In [ ]:
import scanpy as sc

# Load the file
adata = sc.read_h5ad('/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB.h5ad')

In [ ]:
# Extract specific batches
# Specify multiple batches (e.g., 'batch_1' and 'batch_2')
batches_to_extract = ['timecourse_7d_4', 'timecourse_7d_9',]

# Extract cells where the 'batch' column matches any of the specified batches
adata = adata[adata.obs['batch'].isin(batches_to_extract), :]

In [ ]:
adata.write('/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_7d_selected.h5ad')

# Check if the file was actually saved
import os
file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_7d_selected.h5ad'
if os.path.exists(file_path):
    print(f"File successfully written to: {file_path}")
else:
    print(f"Error: File was NOT written to: {file_path}")

In [ ]:
df_meta = pd.DataFrame(data={'Cell':list(adata.obs.index),
                             'cell_type':[ i for i in adata.obs['cell_labels']]
                            })
df_meta.set_index('Cell', inplace=True)
df_meta.to_csv('metadata_7d.tsv', sep = '\t')

In [ ]:
print("Unique values of df_meta['cell_type']:")
print(df_meta['cell_type'].unique())

In [ ]:
cpdb_file_path = '/content/drive/MyDrive/scRNA_analysis/db/cellphonedb.zip'
meta_file_path = '/content/drive/MyDrive/scRNA_analysis/db/metadata_7d.tsv'
counts_file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_7d_selected.h5ad'
out_path = '/content/drive/MyDrive/scRNA_analysis/figures_Ver2'

In [ ]:
import anndata

adata = anndata.read_h5ad(counts_file_path)
adata.shape

In [ ]:
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

cpdb_results = cpdb_statistical_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,             # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                       # optional: whether to score interactions or not.
    iterations = 1000,                               # denotes the number of shufflings performed in the analysis.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 5,                                     # number of threads to use in the analysis.
    debug_seed = 42,                                 # debug randome seed. To disable >=0.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    pvalue = 0.05,                                   # P-value threshold to employ for significance.
    subsampling = False,                             # To enable subsampling the data (geometri sketching).
    subsampling_log = True,                         # (mandatory) enable subsampling log1p for non log-transformed data inputs.
    subsampling_num_pc = 100,                        # Number of componets to subsample via geometric skectching (dafault: 100).
    subsampling_num_cells = 1000,                    # Number of cells to subsample (integer) (default: 1/3 of the dataset).
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = out_path,                          # Path to save results.
    output_suffix = "timecourse_7d"                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

In [ ]:
# timecourse_14d Version

In [ ]:
import scanpy as sc

# Read the file
adata = sc.read_h5ad('/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB.h5ad')
# Extract specific batches
# Specify multiple batches (e.g., 'batch_1' and 'batch_2')
batches_to_extract = ['timecourse_14d_10', 'timecourse_14d_9', 'timecourse_14d_8',]

# Extract cells where the 'batch' column matches any of the specified batches
adata = adata[adata.obs['batch'].isin(batches_to_extract), :]

adata.write('/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_14d_selected.h5ad')

# Check if the file was actually saved
import os
file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_14d_selected.h5ad'
if os.path.exists(file_path):
    print(f"File successfully written to: {file_path}")
else:
    print(f"Error: File was NOT written to: {file_path}")

df_meta = pd.DataFrame(data={'Cell':list(adata.obs.index),
                             'cell_type':[ i for i in adata.obs['cell_labels']]
                            })
df_meta.set_index('Cell', inplace=True)
df_meta.to_csv('metadata_14d.tsv', sep = '\t')
print("Unique values of df_meta['cell_type']:")
print(df_meta['cell_type'].unique())

cpdb_file_path = '/content/drive/MyDrive/scRNA_analysis/db/cellphonedb.zip'
meta_file_path = '/content/drive/MyDrive/scRNA_analysis/db/metadata_14d.tsv'
counts_file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_14d_selected.h5ad'
out_path = '/content/drive/MyDrive/scRNA_analysis/figures_Ver2'

import anndata

adata = anndata.read_h5ad(counts_file_path)
adata.shape
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

cpdb_results = cpdb_statistical_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,             # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                       # optional: whether to score interactions or not.
    iterations = 1000,                               # denotes the number of shufflings performed in the analysis.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 5,                                     # number of threads to use in the analysis.
    debug_seed = 42,                                 # debug randome seed. To disable >=0.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    pvalue = 0.05,                                   # P-value threshold to employ for significance.
    subsampling = False,                             # To enable subsampling the data (geometri sketching).
    subsampling_log = True,                         # (mandatory) enable subsampling log1p for non log-transformed data inputs.
    subsampling_num_pc = 100,                        # Number of componets to subsample via geometric skectching (dafault: 100).
    subsampling_num_cells = 1000,                    # Number of cells to subsample (integer) (default: 1/3 of the dataset).
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = out_path,                          # Path to save results.
    output_suffix = "timecourse_14d"                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

In [ ]:
import scanpy as sc

# Read the file
adata = sc.read_h5ad('/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB.h5ad')
# Extract specific batches
# Specify multiple batches (e.g., 'batch_1' and 'batch_2')
batches_to_extract = ['timecourse_1m_10', 'timecourse_1m_5', 'timecourse_1m_9',]

# Extract cells where the 'batch' column matches any of the specified batches
adata = adata[adata.obs['batch'].isin(batches_to_extract), :]

adata.write('/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_1m_selected.h5ad')

# Check if the file was actually saved
import os
file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_1m_selected.h5ad'
if os.path.exists(file_path):
    print(f"File successfully written to: {file_path}")
else:
    print(f"Error: File was NOT written to: {file_path}")

df_meta = pd.DataFrame(data={'Cell':list(adata.obs.index),
                             'cell_type':[ i for i in adata.obs['cell_labels']]
                            })
df_meta.set_index('Cell', inplace=True)
df_meta.to_csv('metadata_1m.tsv', sep = '\t')
print("Unique values of df_meta['cell_type']:")
print(df_meta['cell_type'].unique())

cpdb_file_path = '/content/drive/MyDrive/scRNA_analysis/db/cellphonedb.zip'
meta_file_path = '/content/drive/MyDrive/scRNA_analysis/db/metadata_1m.tsv'
counts_file_path = '/content/drive/MyDrive/scRNA_analysis/Results_0-1m/Meta_Mix_CPDB_1m_selected.h5ad'
out_path = '/content/drive/MyDrive/scRNA_analysis/figures_Ver2'

import anndata

adata = anndata.read_h5ad(counts_file_path)
adata.shape
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

cpdb_results = cpdb_statistical_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,             # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                       # optional: whether to score interactions or not.
    iterations = 1000,                               # denotes the number of shufflings performed in the analysis.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 5,                                     # number of threads to use in the analysis.
    debug_seed = 42,                                 # debug randome seed. To disable >=0.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    pvalue = 0.05,                                   # P-value threshold to employ for significance.
    subsampling = False,                             # To enable subsampling the data (geometri sketching).
    subsampling_log = True,                         # (mandatory) enable subsampling log1p for non log-transformed data inputs.
    subsampling_num_pc = 100,                        # Number of componets to subsample via geometric skectching (dafault: 100).
    subsampling_num_cells = 1000,                    # Number of cells to subsample (integer) (default: 1/3 of the dataset).
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = out_path,                          # Path to save results.
    output_suffix = "timecourse_14d"                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )